# Analyse des commentaires
Dans ce notebook, nous allons regarder en détail les commentaires laissés par les utilisateurs.
Le travail sera divisé en deux parties : Construction du corpus et Début ? d'analyse des fréquences

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from surprise import NMF, Dataset
from surprise.reader import Reader
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
from nltk.collocations import BigramCollocationFinder, BigramAssocMeasures, TrigramCollocationFinder, TrigramAssocMeasures
from itertools import product
from scipy import sparse

from boardgames_recsys.data.filtering import filter_df
import boardgames_recsys.evaluation.bigrams as bev
import boardgames_recsys.text.filtering as ft
from boardgames_recsys.data.matrix import *
from boardgames_recsys.models.collaborative_filtering import *
from boardgames_recsys.evaluation.ratings import *
from boardgames_recsys.text.lemmatization import *


%load_ext autoreload
%autoreload 2
%matplotlib inline

#### Functions

In [ ]:
def words_freq(data, corpus) -> pd.DataFrame:
    """
    Construction d'un dataframe avec la fréquence des mots dans un corpus
    """

    lem, occurences = np.unique(data['Lemma'], return_counts=True)

    df = pd.DataFrame({'Lemma': lem, 'Freq': occurences})
    nb_comments = data["Comment line"].nunique()
    df['Freq'] = df['Freq'].apply(lambda val: val/nb_comments)

    # Garder uniquement les lemmas qui appraissent dans le corpus
    #return df[df['Lemma'].isin(corpus)]
    return df

def construction_corpus(lemmas:pd.DataFrame, taille: int) -> dict:
    """ 
    Construction d'un corpus à partir d'une BDD de commentaires
    avis.colums = 'Comment title', 'Comment body'

    Retourne df avec mots du corpus et leurs fréquences, les 'taille' plus fréquentes
    """

    # Corpus creation from lemmatized dataframe
    lemmas = lemmas[~lemmas["Lemma"].isna()]
    lemmas = lemmas[lemmas['Part of speech'].isin(['ADJ', 'NOM', "VER", "NEG"])]
    lemmas = lemmas[~lemmas["Lemma"].isin(["bref", "bof", "excelent", "bon", "autre", "seul", "tendre", "fin"
                                           "super", "superbe", "juste", "jouable", "ca", "faire", "pouvoir", "ausi"])]
    lemmas = lemmas['Lemma'].to_numpy()

    # Occurencies calculation for each lemma
    lem, occ = np.unique(lemmas, return_counts=True)
    freq_lem = pd.DataFrame({'lemma': lem, 'freq': occ})

    freq_lem = freq_lem.sort_values(by=['freq'], ascending=False)
    return freq_lem.head(taille)['lemma'].to_numpy()

#### Data

In [ ]:
folder = "../database_cleaned"
avis_clean  = pd.read_csv(f"{folder}/avis_clean.csv", index_col=0)
jeux_clean  = pd.read_csv(f"{folder}/jeux_clean.csv", index_col=0)
users       = pd.read_csv(f"{folder}/users.csv", index_col=0)

min_reviews = 10 
rev_filter = filter_df(avis_clean, min_reviews)
games_means = rev_filter[["Game id", "Rating"]].groupby("Game id").mean().reset_index()

rev_filter = rev_filter.assign(index=np.arange(0, rev_filter.shape[0]))
rev_filter_center, _= center_score(rev_filter)

In [ ]:
rev_filter = rev_filter.reset_index()

### Corpus 5000 mots

In [ ]:
lemmas = pd.read_csv("../generated_data/Lemmas_VER_cleaned.csv", index_col=0)
corpus = construction_corpus(lemmas, 5000) 
lemmas = lemmas[lemmas["Lemma"].isin(corpus)] # only words in corpus

# Joined lemmas
comments = lemmas.groupby("Comment line")["Lemma"].apply(" ".join).reset_index().rename(columns={"Lemma" : "Comment"})

In [ ]:
com = rev_filter[["Game id", "User id", "index"]].merge(lemmas, right_on="Comment line", left_on="index")
com = com.groupby(["Game id", "User id", "Comment line"])["Lemma"].apply(" ".join).reset_index().rename(columns={"Lemma" : "Comment"})

### NMF 20 latent factors

In [ ]:
model = NMF(n_factors=20, random_state=42, biased=False, reg_pu= 0.1, reg_qi= 0.1)
data = Dataset.load_from_df(rev_filter[["User id", "Game id", "Rating"]], reader=Reader(rating_scale=(0, 10)))
trainset = data.build_full_trainset()
nmf = model.fit(trainset)

# Extract matrices
U = nmf.pu  # User-feature matrix (W)
G = nmf.qi  # Item-feature matrix (H)

games_ids = np.array([trainset.to_raw_iid(i) for i in range(len(G))])
users_ids = np.array([trainset.to_raw_uid(u) for u in range(len(U))])
G = G[np.argsort(games_ids), :]

### 30 KMeans games clusters 

In [ ]:
sns.set_theme(rc={"figure.figsize":(6, 5)})
NB_CLUSTERS = 30
kmeans = KMeans(n_clusters=NB_CLUSTERS, random_state=42) 
kmeans.fit(G) 

games_clusters = pd.DataFrame(data={"Game id" : np.sort(games_ids), "Cluster" : kmeans.labels_})

In [ ]:
# Séparation de la bdd 
positifs = rev_filter_center[rev_filter_center['Rating'] >= 0]
negatifs = rev_filter_center[rev_filter_center['Rating'] < 0]

print("Nombre d'avis negatif", len(negatifs)/len(rev_filter_center))
print("Nombre d'avis positif", len(positifs)/len(rev_filter_center))

In [ ]:
fuck[(fuck['User id'] == 201) & (fuck['Game id'] == 10409)]

In [ ]:
fuck = rev_filter[["Game id", "User id", "level_0"]].merge(lemmas, right_on="Comment line", left_on="level_0")
fuck = fuck.groupby(["Comment line", "Game id", "User id"])["Lemma"].apply(" ".join).reset_index().rename(columns={"Lemma" : "Comment"})

In [ ]:
c = rev_filter[["Game id", "User id", "index"]].merge(lemmas, right_on="Comment line", left_on="index")
c = c.groupby(["Comment line", "Game id", "User id"])["Lemma"].apply(" ".join).reset_index().rename(columns={"Lemma" : "Comment"})

In [ ]:
c[(c["User id"]==201) & (c["Game id"]==10409)]

In [ ]:
rev_filter[(rev_filter["User id"] == 201) & (rev_filter["Game id"] == 10409)]

In [ ]:
# filter rev filter that is in c, filtered lemma by corpus

filtered_rev_df = rev_filter.merge(c, on=["Game id", "User id"], how="inner")

In [ ]:
filtered_rev_df[(filtered_rev_df["Game id"] == 6) & (filtered_rev_df["User id"] == 3380)]['Comment body'].values

In [ ]:
lemmas_pos = positifs[["Game id", "User id", "index"]].merge(lemmas, right_on="Comment line", left_on="index")
lemmas_neg = negatifs[["Game id", "User id", "index"]].merge(lemmas, right_on="Comment line", left_on="index")
lemmas_pos = lemmas_pos.drop(["index"], axis=1)
lemmas_neg = lemmas_neg.drop(["index"], axis=1)

lemmas_all = rev_filter[["User id", "Game id", "index"]].merge(lemmas, right_on="Comment line", left_on="index")
lemmas_all = lemmas_all.drop(["index"], axis=1)

In [ ]:
comments_neg = lemmas_neg.groupby(by=["Comment line", "Game id", "User id"])["Lemma"].apply(" ".join).reset_index()
comments_neg = comments_neg.assign(pos_neg = "negative")

comments_pos = lemmas_pos.groupby(by=["Comment line", "Game id", "User id"])["Lemma"].apply(" ".join).reset_index()
comments_pos = comments_pos.assign(pos_neg = "positive")

comments_all = pd.concat([comments_neg, comments_pos])
comments_all_count = comments_all[["Game id", "pos_neg", "User id"]].groupby(["Game id", "pos_neg"]).count().rename(columns={"User id":"count"}).reset_index()

In [ ]:
# filters user id that have at least 10 reviews

count_us = comments_all[["User id", "Lemma"]].groupby("User id").count().sort_values(ascending=False, by='Lemma')
users_keeps = count_us[count_us['Lemma'] >= 10].index
users_keeps

In [ ]:
comments_all = comments_all[comments_all['User id'].isin(users_keeps)]
comments_all = comments_all.sort_values(by=['Game id', 'User id']).reset_index(drop=True)
comments_all

In [ ]:
filtered_rev_df[filtered_rev_df["User id"]== 0][['Comment body', 'Comment']].head(1).values

In [ ]:
filtered_rev_df = filtered_rev_df[filtered_rev_df['User id'].isin(users_keeps)]
filtered_rev_df = filtered_rev_df.sort_values(by=['Game id', 'User id']).reset_index(drop=True)
filtered_rev_df

In [ ]:
rev_neg_count = comments_neg["Game id"].value_counts().reset_index()
rev_pos_count = comments_pos["Game id"].value_counts().reset_index()

# Filter games so that each game has at least 10 pos and 10 neg reviews
games_preserved = rev_neg_count[rev_neg_count["Game id"].isin(rev_pos_count.loc[rev_pos_count["count"] >= 10, "Game id"])
                                & rev_neg_count["Game id"].isin(rev_neg_count.loc[rev_neg_count["count"] >= 10, "Game id"])]["Game id"].values
                                
mask = np.isin(np.sort(games_ids), games_preserved)

# Games clusters contains only games that were filtered
games_clusters = pd.DataFrame(data={"Game id":np.sort(games_ids)[mask], "Cluster":kmeans.labels_[mask]})

In [ ]:
# Barplot the distribution of pos/neg comments
def plot_pos_neg_games(selected_games, comments_all_count, title, all=False):
    sns.set_theme(rc={"figure.figsize":(15, 6)})
    filtered = comments_all_count[comments_all_count["Game id"].isin(selected_games["Game id"])]
    if not all:
        filtered = filtered.head(50)
    ax = sns.barplot(data=filtered, x="Game id", y="count", hue="pos_neg", errorbar=None)
    ax.set_title(title)
    if all:
        ax.set(xticklabels=[])

def create_df(ngram_finder, ngram_stat):
        bigram_freq = ngram_finder.score_ngrams(ngram_stat)

        bigrams_df = pd.DataFrame(data=[list(info) for info in bigram_freq])
        bigrams_df[0] = bigrams_df[0].apply(list).apply(" ".join)
        bigrams_df = bigrams_df.rename(columns={0:"Lemma", 1:"Freq"})
        return bigrams_df

def get_Ngrams(game, ngram_finder, ngram_stat):
    comments_pos = lemmas_pos[lemmas_pos["Game id"] == game].groupby("Comment line")["Lemma"].apply(list)
    comments_neg = lemmas_neg[lemmas_neg["Game id"] == game].groupby("Comment line")["Lemma"].apply(list)
    
    #if comments_pos.size > 0:
    bigram_finder_pos = ngram_finder.from_documents(comments_pos)
    freq_pos = create_df(bigram_finder_pos, ngram_stat)
    #if comments_neg.size > 0:
    bigram_finder_neg = ngram_finder.from_documents(comments_neg)
    freq_neg = create_df(bigram_finder_neg, ngram_stat)
    
    diff_freq = ft.diff_freq(freq_pos, freq_neg)

    return freq_pos, freq_neg, diff_freq

def plot_games_Ngrams_freq_diff(selected_games:np.array, nrows:int, ncols:int, figsize:tuple, ngram_finder, ngram_stat, games_means):
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    for game, (i, j) in zip(selected_games, list(product(range(0, nrows - 1, 2), range(ncols)))):
        mean = games_means[games_means["Game id"] == game]["Rating"].item()

        _, _, diff_check_games = get_Ngrams(game, ngram_finder,ngram_stat)

        sns.barplot(data=diff_check_games.head(20), y="Lemma", x="Freq differency", ax=axes[i, j])
        sns.barplot(data=diff_check_games.tail(20), y="Lemma", x="Freq differency", ax=axes[i + 1, j])

        axes[i, j].set_title(f"Game {game} head freq_diff {mean:.2f}")
        axes[i + 1, j].set_title(f"Game {game} tail freq_diff {mean:.2f}")

    plt.tight_layout()

def plot_games_Ngrams_all(selected_games:np.array, nrows:int, ncols:int, figsize:tuple, ngram_finder, ngram_stat, games_means):
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    for game, (i, j) in zip(selected_games, list(product(range(0, nrows, 2), range(ncols)))):
        
        mean = games_means[games_means["Game id"] == game]["Rating"].item()
        pos, neg, _ = get_Ngrams(game, ngram_finder,ngram_stat)

        sns.barplot(data=pos.head(20), y="Lemma", x="Freq", ax=axes[i, j])
        sns.barplot(data=neg.head(20), y="Lemma", x="Freq", ax=axes[i + 1, j])

        axes[i, j].set_title(f"Game {game} head pos {mean:.2f}")
        axes[i + 1, j].set_title(f"Game {game} head neg {mean:.2f}")

    plt.tight_layout()

In [ ]:
comments_all[(comments_all['User id'] == 201)&(comments_all['Game id'] == 10409)]

In [ ]:
avis_clean

In [ ]:
fuck = fuck.merge(avis_clean[['Rating', 'Game id','User id']],on =['Game id','User id'], how='left')
fuck

In [ ]:
filtered_rev_df[(filtered_rev_df['User id'] == 201)&(filtered_rev_df['Game id'] == 10409)]

In [ ]:
# Init
# matrix_ratings, mask_ratings, users_table, games_table = get_matrix_user_game(rev_filter)
matrix_ratings, mask_ratings, users_table, games_table = get_matrix_user_game(fuck)
cos_sim_matrix = calc_distance_matrix(matrix_ratings, mask_ratings, "cos")
top_users = filtered_rev_df[["User id", "Rating"]].groupby("User id").count().reset_index().sort_values(by="Rating", ascending=False)["User id"].values

games_to_consider = games_clusters["Game id"].values
users_mean = filtered_rev_df[["User id", "Rating"]].groupby("User id").mean().reset_index()

---

In [ ]:
comments_all

### Using tf idf to filter bigrams

In [ ]:
fuck

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# lemmatized comments
all_doc = fuck['Comment']
vectorizer = TfidfVectorizer(ngram_range=(2, 2), min_df=5, max_df=0.8) # bigrams
vectors = vectorizer.fit_transform(all_doc)

In [ ]:
bigrams_ens = vectorizer.get_feature_names_out()

In [ ]:
comments_all = comments_all.drop(columns=['Comment line']).reset_index()
comments_all = comments_all.drop(columns=['index']).reset_index()

In [ ]:
# plotting for the threshold
tfidf_value = vectors.data

In [ ]:
np.random.seed(1)
bev._knn_sim(208, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, k=40)

In [ ]:
# top 50 userss
top50 = []

# np.random.seed(1)
for id in top_users[:50]:
    top50.append(bev.knn_ROUGE(id, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, threshold = -1, k = 40, topx = None))

In [ ]:
# Create DataFrame
df_top50 = pd.DataFrame(top50, columns=['Similar', 'Random', 'Distant'])
df_top50.insert(0, 'User id', top_users[:50])

In [ ]:
print(df_top50[['Similar', 'Random', 'Distant']].mean())

plt.figure(figsize=(8, 6))

df_melted = df_top50.melt(id_vars='User id', value_vars=['Distant', 'Random', 'Similar'], 
                    var_name='variable', value_name='value')
df_melted['value'] = df_melted['value'] * 100

group_means = df_melted.groupby('variable')['value'].mean()

# Overlay means as black diamonds
for i, (group, mean) in enumerate(group_means.items()):
    print(group)
    plt.scatter(i, mean, color='black', marker='o', zorder=10, label="Mean" if i == 0 else None)

sns.violinplot(x='variable', y='value', data=df_melted, hue='variable', inner='box', cut=0, fill=False)

plt.title("ROUGE-2 Score for well rated games, 50 most active users, k=40")
plt.ylabel("ROUGE-2 Score (in %)")
plt.xlabel("Type of users")
plt.legend(loc='center left', bbox_to_anchor=(0.9, 0.8))
plt.tight_layout()

# Plot violin plot
plt.savefig("Rouge_most_active_no_threshold.svg")

In [ ]:
comments_all[comments_all["Game id"]==306]

In [ ]:
a = []

np.random.seed(1)
for id in top_users[:50]:
    a.append(bev.knn_ROUGE(id, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, threshold = 0.13, k = 40, topx = None))

In [ ]:
# Create DataFrame
df = pd.DataFrame(a, columns=['Similar', 'Random', 'Distant'])
df.insert(0, 'User id', top_users[:50])

In [ ]:
print(df[['Similar', 'Random', 'Distant']].mean())

plt.figure(figsize=(8, 6))

df_melted = df.melt(id_vars='User id', value_vars=['Similar', 'Random', 'Distant'], 
                    var_name='variable', value_name='value')
df_melted['value'] = df_melted['value'] * 100

group_means = df_melted.groupby('variable')['value'].mean()

# Overlay means as black diamonds
for i, (group, mean) in enumerate(group_means.items()):
    print(group)
    plt.scatter(i, mean, color='black', marker='o', zorder=10, label="Mean" if i == 0 else None)

sns.violinplot(x='variable', y='value', data=df_melted, hue='variable', inner='box', cut=0, fill=False, order=['Distant', 'Random', 'Similar'])

plt.title("ROUGE-2 Score for well rated games, 50 most active users, k=40, threshold=0.13")
plt.ylabel("ROUGE-2 Score (in %)")
plt.xlabel("Type of users")
plt.legend(loc='center left', bbox_to_anchor=(0.9, 0.8))
plt.tight_layout()
plt.savefig("Rouge_most_active_threshold.svg")
plt.show()

In [ ]:
rand_200 = []
users_kept = []

np.random.seed(1)
random_users = np.random.choice(users_keeps, size=200, replace=False)

for id in random_users:
    res = bev.knn_ROUGE(id, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, threshold = -1, k = 40, topx = 150)
    if res is not None:
        rand_200.append(res) 
        users_kept.append(id)

In [ ]:
# Create DataFrame
df_200 = pd.DataFrame(rand_200, columns=['Similar', 'Random', 'Distant'])
df_200.insert(0, 'User id', users_kept)
df_200

In [ ]:
df_200[['Similar', 'Random', 'Distant']].mean()

In [ ]:
plt.figure(figsize=(8, 6))

df_melted = df_200.melt(id_vars='User id', value_vars=['Distant', 'Random', 'Similar'], 
                    var_name='variable', value_name='value')
df_melted['value'] = df_melted['value'] * 100

group_means = df_melted.groupby('variable')['value'].mean()

# Overlay means as black diamonds
for i, (group, mean) in enumerate(group_means.items()):
    print(group)
    plt.scatter(i, mean, color='black', marker='o', zorder=10, label="Mean" if i == 0 else None)


sns.violinplot(x='variable', y='value', data=df_melted, hue='variable', inner='box', cut=0, fill=False, order=['Distant', 'Random', 'Similar'])

plt.title("ROUGE Score bigrams for well rated games, 200 random users, k=40, topx=150")
plt.ylabel("ROUGE Score (in %)")
plt.xlabel("Type of users")
plt.legend(loc='center left', bbox_to_anchor=(0.9, 0.8))
plt.tight_layout()

# Plot violin plot
plt.savefig("Rouge_random_no_threshold_top150.svg")

With threshold 0.13

In [ ]:
fuck

In [ ]:
a = []
users_kept = []

np.random.seed(1)
# random_users = np.random.choice(users_keeps, size=200, replace=False)

for id in top_users[:50]:
    res = bev.knn_ROUGE(id, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, fuck, vectors, bigrams_ens, threshold = 0.13, k = 40, topx = None)
    if res[0] is not None:
        a.append(res) 
        users_kept.append(id)

In [ ]:
rouge_1 = [t[0] for t in a]  # list of tuples (1,2,3), (7,8,9), ...
rouge_2 = [t[1] for t in a]

# Create DataFrame
df = pd.DataFrame({
    'rouge_1_similar': [x[0] for x in rouge_1],
    'rouge_1_random': [x[1] for x in rouge_1],
    'rouge_1_distant': [x[2] for x in rouge_1],
    'rouge_2_similar': [x[0] for x in rouge_2],
    'rouge_2_random': [x[1] for x in rouge_2],
    'rouge_2_distant': [x[2] for x in rouge_2],
})

In [ ]:
df.rename(columns={
    'rouge_1_similar': 'Similar',
    'rouge_1_random': 'Random',
    'rouge_1_distant': 'Distant'
}, inplace=True)

In [ ]:
df

In [ ]:
df.insert(0, 'User id', users_kept)
df.set_index("User id")

In [ ]:
# # Create DataFrame
# df = pd.DataFrame(a, columns=['Similar', 'Random', 'Distant'])
# df.insert(0, 'User id', users_kept)
# df.set_index("User id")

In [ ]:
df[['Similar', 'Random', 'Distant']].mean()

In [ ]:
from matplotlib.patches import Patch
import matplotlib.lines as mlines

plt.figure(figsize=(8, 6))

df_melted = df.melt(id_vars='User id', value_vars=['Similar', 'Random', 'Distant'], 
                    var_name='variable', value_name='value')
df_melted['value'] = df_melted['value'] * 100
df_melted['type'] = df_melted['variable'] 
group_means = df_melted.groupby('variable')['value'].mean()

palette = sns.color_palette()  # or your custom palette
hue_categories = df_melted['variable'].unique()
patches = [Patch(edgecolor=palette[i], facecolor='none', label=cat, linewidth=1.5) for i, cat in enumerate(hue_categories)]

sns.violinplot(x='variable', y='value', data=df_melted, hue='type', inner='box', cut=0, fill=False, order=["Distant", "Random", "Similar"])

lab = ['Mean Similar', 'Mean Random', 'Mean Less Similar']
for i, (group, mean) in enumerate(group_means.items()):
    plt.scatter(i, mean, color='black', zorder=10, label="Mean" if i==0 else None)

mean_handle = mlines.Line2D([], [], color='black', marker='o', linestyle='None', markersize=7, label='Mean')

# Combine patch handles and the mean handle
all_handles = patches + [mean_handle]

plt.title("ROUGE Score bigrams for well rated games, 200 random users, k=40, treshold=0.13")
plt.ylabel("ROUGE Score (in %)")
plt.xlabel("Type of users")
plt.legend(handles=all_handles, loc='center left', bbox_to_anchor=(1, 0.8))


# plt.savefig("Rouge_random_threshold.svg")
plt.tight_layout()
plt.show()

Annexe plot for 50 users

In [ ]:
my_list = [83, 91, 92, 94, 291, 455, 574, 885, 934, 1816, 1899, 1900, 1906, 1998, 2994]

In [ ]:
df_longR1 = scores_all_top[scores_all_top['Pred by'] == 'R1'].melt(id_vars=['User id'], 
                  value_vars=['Similar', 'Random', 'Distant'],
                  var_name='type', 
                  value_name='values')

# Step 2: Explode the lists into rows
df_longR1 = df_longR1.explode('values')

In [ ]:
df_longR2 = scores_all_top[scores_all_top['Pred by'] == 'R2'].melt(id_vars=['User id'], 
                  value_vars=['Similar', 'Random', 'Distant'],
                  var_name='type', 
                  value_name='values')

# Step 2: Explode the lists into rows
df_longR2 = df_longR2.explode('values')

In [ ]:
avis_clean[avis_clean['Username'] =='Dexter269']

In [ ]:
df_longR1 = df_longR1[df_longR1['User id'].isin(rand_list)].dropna()

In [ ]:
df_longR2 = df_longR2[df_longR2['User id'].isin(rand_list)].dropna()

In [ ]:
df_longR1['type'] = df_longR1['type'].replace({
    'Similar': 'Similar users',
    'Random': 'Random users',
    'Distant': 'Distant users'
})

In [ ]:
np.random.seed(1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
# top_users_sample = np.random.choice(scores_all_top["User id"].unique(), size=15)
# df_top = scores_all_top[scores_all_top["User id"].isin(my_list)]
# df_top = df_top[df_top["Pred by"] == "R1"] 
# print(df_top)

sns.stripplot(data=df_longR1, x="User id", y="values", hue="type", jitter=True, dodge=False, ax=ax1)
sns.stripplot(data=df_longR2, x="User id", y="values", hue="type", jitter=True, dodge=False, ax=ax2)

ax1.set_ylabel("ROUGE-1 (in %)")
ax2.set_ylabel("ROUGE-2 (in %)")
ax1.legend(title="Prediction by")
ax2.legend(title="Prediction by")

ax1.set_ylim(-4, 105)
ax2.set_ylim(-1, 32)

fig.suptitle("ROUGEs scores distribution for most active users (15 out of 50)", y=0.95)
plt.tight_layout()
plt.savefig("jitter_top_R12_big.svg")

In [ ]:
fuck['User id'].unique()

In [ ]:
# scores_all = pd.DataFrame(columns=['User id', 'Similar', 'Random', 'Distant'])

# nb = 0
# np.random.seed(1)
# random_users = np.random.choice(fuck['User id'].unique(), size=200, replace=False)

# for id in top_users[:50]:
#     scores_user = bev.knn_ROUGE_annexe(id, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, fuck, vectors, bigrams_ens, threshold = 0.13, k = 40, topx = 150)
#     if scores_user is None:
#         continue
#     nb += 1
#     user_df = pd.DataFrame(list(zip(*scores_user)), columns=['Similar', 'Random', 'Distant'])
#     user_df['User id'] = id
#     scores_all = pd.concat([scores_all, user_df])
# print(nb)

In [ ]:
# random
scores_all_rand = scores_all

In [ ]:
# top 50
scores_all_top = scores_all

In [ ]:
type(np.array([(None),(None)])[0])

In [ ]:
scores_all = pd.DataFrame(columns=['User id', 'Pred by', 'Similar', 'Random', 'Distant'])

nb = 0
np.random.seed(1)
# random_users = np.random.choice(fuck['User id'].unique(), size=200, replace=False)

for id in my_list:
    scores_user = bev.knn_ROUGE_annexe(id, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, fuck, vectors, bigrams_ens, threshold = 0.13, k = 40, topx = 150)
    R2, R1 = scores_user
    R2, R1 = R2.tolist(), R1.tolist()
    nb += 1
    user_df = pd.DataFrame([
    [id, 'R1'] + R1,
    [id, 'R2'] + R2,
    ], columns=['User id', 'Pred by', 'Similar', 'Random', 'Distant'])
    scores_all = pd.concat([scores_all, user_df])
print(nb)

In [ ]:
scores_all

In [ ]:
scores_all[scores_all['Pred by']=='R1']

In [ ]:
rand_list = [5014, 2308, 2295, 7263, 3138, 4333, 2442,  886, 1972, 6514, 1774, 2198, 2706, 4595, 5012]

In [ ]:
plt.figure(figsize=(20, 6))

np.random.seed(1)
random_50 = np.random.choice(scores_all['User id'].unique(), 20)
chosen_50 = scores_all[scores_all['User id'].isin(random_50)]

chosen_50['User id'] = chosen_50['User id'].astype(str)
chosen_50[['Similar', 'Random', 'Distant']] = chosen_50[['Similar', 'Random', 'Distant']].astype(int)

df_melted = chosen_50.melt(
    id_vars='User id',
    value_vars=['Similar', 'Random', 'Distant'],
    var_name='Type',
    value_name='Score'
)

# mean_scores = chosen_50.groupby('User id')[['Similar', 'Random', 'Distant']].mean().reset_index()

# sns.lineplot(data=mean_scores, x="User id", y='Similar', color='black', label='Mean', marker='o', sizes=15, alpha=0.6)
sns.stripplot(data=df_melted, x="User id", y='Score',hue='Type', alpha=0.8, size=5)

plt.xticks(fontsize=8)
plt.xlabel('User id')
plt.ylabel('Rouge-2 score (in %)')
plt.title('ROUGE-2 Score on well predicted games, for 20 random users, k=40')
plt.legend()

# plt.savefig('ROUGE2_random_annexe(20)bis.svg')
plt.show()

In [ ]:
avis_clean[(avis_clean['User id'] == 201) & (avis_clean['Game id'] == 10409)]

In [ ]:
import ollama
from nltk import bigrams

ref1 = avis_clean[(avis_clean['User id'] == 0) & (avis_clean['Game id'] == 6179)]["Comment body"].values[0] # rated 8.0
hyp1 = comments_all[(comments_all['User id'] == 0)& (comments_all['Game id'] == 6179)]['Lemma'].values[0]

ref2 = avis_clean[(avis_clean["User id"] == 2258)& (avis_clean["Game id"] == 4118)]["Comment body"].values[0] # rated 2.0
hyp2 = comments_all[(comments_all["User id"] == 2258) & (comments_all["Game id"] == 4118)]['Lemma'].values[0]

def generate_summary_from_bigrams(bigram_list, model='llama3.2:1b'):
    
    messages = [
        {
            "role": "system",
            "content": (
                "You will be given a list of bigrams that correspond to the words of a player's comment for a game. "
                "You have to write a summary about the game as if you were that player, in 15 sentences max. "
                "The comment should keep the relevent information from the bigrams, and is in french."
            )
        },
        {"role": "user", "content": str(list(bigrams(hyp1.split())))},
        {"role": "assistant", "content": ref1},
        {"role": "user", "content": str(list(bigrams(hyp2.split())))},
        {"role": "assistant", "content": ref2},
        {"role": "user", "content": str(bigram_list)}
    ]

    # Call the model
    response = ollama.chat(model=model, messages=messages)

    return response['message']['content']


In [ ]:
import ollama
from nltk import bigrams

ref1 = avis_clean[(avis_clean['User id'] == 0) & (avis_clean['Game id'] == 6179)]["Comment body"].values[0] # rated 8.0
hyp1 = comments_all[(comments_all['User id'] == 0)& (comments_all['Game id'] == 6179)]['Lemma'].values[0]

ref2 = avis_clean[(avis_clean["User id"] == 2258)& (avis_clean["Game id"] == 4118)]["Comment body"].values[0] # rated 2.0
hyp2 = comments_all[(comments_all["User id"] == 2258) & (comments_all["Game id"] == 4118)]['Lemma'].values[0]

def generate_summary_from_bigrams(bigram_list, model='llama3.2:1b'):
    
    messages = [
        {
            "role": "system",
            "content": (
                "You will be given a list of words that correspond to the words of a player's comment for a game. "
                "You have to write a summary about the game as if you were that player, in 15 sentences max. "
                "The comment should keep the relevent information from the bigrams, and is in french."
            )
        },
        {"role": "user", "content": str(list(hyp1.split()))},
        {"role": "assistant", "content": ref1},
        {"role": "user", "content": str(list(hyp2.split()))},
        {"role": "assistant", "content": ref2},
        {"role": "user", "content": str(bigram_list)}
    ]

    # Call the model
    response = ollama.chat(model=model, messages=messages)

    return response['message']['content']


In [ ]:
fuck

In [ ]:
fuck = fuck.reset_index(names="index")

In [ ]:
np.sum(scores_user)/len(scores_user)

In [ ]:
np.random.seed(23)
scores_user = bev.knn_ROUGE(201, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, fuck, vectors, bigrams_ens, threshold = 0.13, k = 40, topx = 150)
scores_user

In [ ]:
np.random.seed(23)
scores_user = bev.knn_ROUGE_annexe(201, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, fuck, vectors, bigrams_ens, threshold = 0.13, k = 40, topx = 150)
scores_user

In [ ]:
scores_user.mean(axis=1)

In [ ]:
sim_g = generate_summary_from_bigrams(user_sim)
sim_g

In [ ]:
unig_llm

In [ ]:
rand_g = generate_summary_from_bigrams(unig_llm)
rand_g

In [ ]:
dist_g = generate_summary_from_bigrams(user_dist)
dist_g

In [ ]:
avis_clean[(avis_clean["Game id"]== 10409) & (avis_clean["User id"]== 201)]['Comment body'].values[0]

In [ ]:
rev_filter_center
rev_filter_center[(rev_filter_center["Game id"]== 10409) & (rev_filter_center["User id"]== 201)]['Comment body'].values[0]

In [ ]:
import textwrap
print("For topx=150")
print("User's comment:")
print("\n".join(textwrap.wrap(fuck[(fuck["Game id"]== 10409) & (fuck["User id"]== 201)]['Comment'].values[0], width=150)))
print("Generated text: similar user")
print("\n".join(textwrap.wrap(sim_g, width=150)))
print("Generated text: random user")
print("\n".join(textwrap.wrap(rand_g, width=100)))
print("Generated text: distant user")
print("\n".join(textwrap.wrap(dist_g, width=100)))

In [ ]:
user_sim = [('apres', 'partie'), ('depart', 'petit'), ('etre', 'depart'), ('faim', 'apres'),
 ('jeu', 'laisser'), ('laisser', 'faim'), ('parer', 'etre'), ('partie', 'decevoir'),
 ('petit', 'jeu')]

user_rand = [('basique', 'jouer'), ('basique', 'pas'), ('chose', 'rester'), ('grand', 'chose'),
 ('jeu', 'de'), ('pas', 'grand'), ('pas', 'theme'), ('rester', 'ultra'),
 ('theme', 'pas'), ('ultra', 'basique')]

user_dist = [('accessible', 'rapide'), ('oublier', 'etre'), ('voir', 'boite'), ('tres', 'facile'),
 ('stop', 'tres'), ('sortir', 'qualite'), ('simple', 'genre'), ('rien', 'main'),
 ('rapide', 'sortir'), ('qualite', 'defaut'), ('perdre', 'avoir'), ('pas', 'dame'),
 ('oublier', 'si'), ('jeu', 'stop'), ('avoir', 'perdre'), ('jeu', 'accessible'),
 ('facile', 'acces'), ('etre', 'cher'), ('dice', 'jeu'), ('debuter', 'soiree'),
 ('dame', 'chance'), ('cher', 'voir'), ('chance', 'lance'), ('base', 'avoir'),
 ('bang', 'mort'), ('avoir', 'rien'), ('zombie', 'dice')]

In [ ]:
unig_llm = ['jeu', 'boite', 'si', 'ne', 'pas', 'chance', 'defaut', 'dice', 'stop', 'tres',
 'cerveau', 'zombie', 'genre', 'de']


In [ ]:
comments_all[(comments_all["Game id"]== 10409) & (comments_all["User id"]== 201)]['Lemma'].values[0]

In [ ]:
scores_all = pd.DataFrame(columns=['User id', 'Similar', 'Random', 'Distant'])
nb = 0

np.random.seed(1)
random_users = np.random.choice(users_keeps, size=200, replace=False)

for id in random_users:
    scores_user = bev.knn_ROUGE_annexe(id, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, threshold = 0.13, k = 40, topx = 150)
    if scores_user is None:
        continue
    nb += 1
    user_df = pd.DataFrame(list(zip(*scores_user)), columns=['Similar', 'Random', 'Distant'])
    user_df['User id'] = id
    scores_all = pd.concat([scores_all, user_df])
print(nb)

In [ ]:
# user 1821 rouge 30% 
vois1821 = [
    'bien chose', 'rare jeu', 'bruire de', 'de boite', 'empeche pas', 'jeu bas',
    'jeu sympathique', 'jouer anglais', 'ludique ne', 'manche apres', 'mot venir',
    'ne empeche', 'occuper bien', 'pas rester', 'temps bien', 'bruire rien',
    'temps mort', 'travailler vocabulaire', 'venir esprit', 'adorer petit',
    'ideal initier', 'initier jeu', 'jeu mot', 'jeu rapide', 'minute vite',
    'petit jeu', 'rapide minute', 'scrabble adorer', 'boite plastique',
    'bien temps', 'bien rapide', 'bien classique', 'chose difficile', 'chose mot',
    'de ne', 'difficile trouver', 'etre repetitif', 'etre vrai', 'gagner limite',
    'hasard bien', 'jeu lettre', 'jouer gagner', 'lettre trouver', 'mauvais jeu',
    'meme mot', 'mot lettre', 'mot revenir', 'pas retomber', 'placement si',
    'possible hasard', 'repetitif possible', 'rester mauvais', 'rien etre',
    'scrabble jeu', 'trouver chose', 'vrai aimer', 'apres aimer', 'vite ideal'
]

hyp1821= [tuple(bigram.split()) for bigram in vois1821[:100]]
ref1821 = avis_clean[(avis_clean['User id'] == 1821) & (avis_clean['Game id'] == 4451)]['Comment body'].values[0]
ref1821

In [ ]:
generate_summary_from_bigrams(hyp1821[:100])

In [ ]:
scores_all.set_index('User id', inplace=True)
scores_all.idxmax()

In [ ]:
scores_all[scores_all.index == 1821]

In [ ]:
scores_all[scores_all['Similar'] == 30]

In [ ]:
scores_all = pd.DataFrame(columns=['User id', 'Similar', 'Random', 'Distant'])
nb = 0

np.random.seed(1)
for id in top_users[:50]:
    scores_user = bev.knn_ROUGE_annexe(id, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, threshold = 0.13, k = 40, topx = 150)
    if scores_user is None:
        continue
    nb += 1
    user_df = pd.DataFrame(list(zip(*scores_user)), columns=['Similar', 'Random', 'Distant'])
    user_df['User id'] = id
    scores_all = pd.concat([scores_all, user_df])
print(nb)

In [ ]:
scores_all.max() 

In [ ]:
scores_all[scores_all["User id"] == 1903]

1903 simi, 1191 dist comms

In [ ]:
# user 1191 25% rouge 2
vois1191 = [
    'adorer jeu', 'phase negociation', 'avoir adversaire', 'pret lancer',
    'mettre difficile', 'joueur pret', 'joueur aimer', 'jeu cool',
    'difficile trouver', 'cool petit', 'bien jeu', 'aimer bien', 'vite petit',
    'trouver joueur', 'un partie', 'temps mauvais', 'reproche jeu', 'petit avoir',
    'permettre immersion', 'partie grand', 'partie devenir', 'oublier vite',
    'tric trac', 'jeu partie', 'annee ne', 'jeu conquete', 'temps joueur',
    'temps jeu', 'strategie demander', 'ne laisser', 'meme temps',
    'laisser place', 'joueur falloir', 'jeu strategie', 'incontournable jeu',
    'negociation temps', 'falloir posseder', 'demander temps', 'conquete ne',
    'cela meme', 'pas neophyte', 'partie memorable', 'motiver jouer',
    'jouer sept', 'noter durer', 'ne confiance', 'jeu sublime', 'agression jeu',
    'jouer jour', 'jeu jouer', 'jeu favori', 'falloir pas', 'cela humeur',
    'cela dire', 'avoir temps', 'allier trahir', 'penible jeu', 'monter un',
    'pas fun', 'ordre bien', 'jeu jeu', 'jeu diplo', 'bien ne', 'regret ne',
    'pas avoir', 'must jeu', 'jour si', 'ne falloir', 'pas reunir',
    'personne avoir', 'mettre noter', 'mauvais ambiance', 'limiter phase',
    'lieu partie', 'jeu prenant', 'jeu experience', 'jeu empecher',
    'immersion total', 'experience unique', 'empecher mettre',
    'devenir interminable', 'confiance personne', 'avoir lieu',
    'ambiance generer', 'trahir cela', 'temps libre', 'reunir personne',
    'voir cela'
]

hyp1191= [tuple(bigram.split()) for bigram in vois1191[:100]]
ref1191 = avis_clean[(avis_clean['User id'] == 1191) & (avis_clean['Game id'] == 6332)]['Comment body'].values[0]

# user 1903 22% score rouge
vois1903 = [
    'jouer cluedo', 'cluedo jeu', 'deduction jeu', 'compliquer jeu',
    'exister petit', 'etre demasquer', 'dernier ne', 'court heure',
    'continuer jouer', 'arriver certain', 'cela aimer', 'si vouloir', 'an petit',
    'aimer decouvrir', 'ne importer', 'vouloir developper', 'femme partie',
    'importer moment', 'importer ne', 'manquer richesse', 'moment personne',
    'monde sentir', 'mystere abbaye', 'ne longue', 'partie renouveler',
    'pas abuser', 'petit femme', 'petit regle', 'petit soeur', 'regle compliquer',
    'ressortir jouer', 'richesse partie', 'sentir concerner', 'verite si',
    'revanche si', 'si utiliser', 'heure reflechir', 'jeu jouer', 'jouer horreur',
    'jouer joueur', 'jouer partie', 'joueur ne', 'ne prenant', 'pas tete',
    'prenant pas', 'reflechir jouer', 'tete jouer', 'aimer pas', 'bien bien',
    'bien sur', 'carte joueur', 'distribution carte', 'pas systeme',
    'retour source', 'systeme deduction', 'systeme deplacement', 'tourner bien',
    'trouver information', 'utiliser variant', 'variant partie', 'jeu logique',
    'avoir jouer', 'dire falloir', 'excellent approche', 'falloir avoir',
    'penser excellent', 'soeur an'
]

hyp1903= [tuple(bigram.split()) for bigram in vois1903[:100]]
ref1903 = avis_clean[(avis_clean['User id'] == 1903) & (avis_clean['Game id'] == 1752)]['Comment body'].values[0]

In [ ]:
sum1191 = generate_summary_from_bigrams(hyp1191[:100])
sum1191

In [ ]:
sum1903 = generate_summary_from_bigrams(hyp1903[:100])
sum1903

In [ ]:
import textwrap
print("For topx=150")
print("Generated text:")
print("\n".join(textwrap.wrap(sum1191, width=100)))
print("\nUser's comment:")
print("\n".join(textwrap.wrap(ref1191, width=100)))

In [ ]:
print("For topx=150")
print("Generated text:")
print("\n".join(textwrap.wrap(sum1903, width=100)))
print("\nUser's comment:")
print("\n".join(textwrap.wrap(ref1903, width=100)))

In [ ]:
import textwrap
print("For topx=100")
print("Generated text:")
print("\n".join(textwrap.wrap(sum1191, width=100)))
print("\nUser's comment:")
print("\n".join(textwrap.wrap(ref1191, width=100)))

In [ ]:
print("For topx=100")
print("Generated text:")
print("\n".join(textwrap.wrap(sum1903, width=100)))
print("\nUser's comment:")
print("\n".join(textwrap.wrap(ref1903, width=100)))

In [ ]:
plt.figure(figsize=(20, 6))

np.random.seed(1)
chosen = np.random.choice(top_users[:50], 50, replace=False)

chosen_df = scores_all[scores_all['User id'].isin(chosen)]
chosen_df['User id'] = chosen_df['User id'].astype(str)
chosen_df[['Similar', 'Random', 'Distant']] = chosen_df[['Similar', 'Random', 'Distant']].astype(int)

df_melted = chosen_df.melt(
    id_vars='User id',
    value_vars=['Similar', 'Random', 'Distant'],
    var_name='Type',
    value_name='Score'
)

sns.stripplot(data=df_melted, x='User id', y='Score', hue="Type", s=4, alpha=0.8)

plt.xticks(fontsize=8)
plt.xlabel('User id')
plt.ylabel('Rouge-2 score (in %)')
plt.title('ROUGE-2 Score on well predicted games, for 50 most active users, k=40')
plt.legend()

plt.savefig("ROUGE2_most_active_annexe.svg")
plt.show()

In [ ]:
l200 = []
users_kept = []

np.random.seed(1)
random_users = np.random.choice(users_keeps, size=200, replace=False)

for id in random_users:
    res = bev.knn_ROUGE(id, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, threshold = 0.13, k = 40, topx = 150)
    if res[0] is not None:
        l200.append(res[0]) 
        users_kept.append(id)
    

In [ ]:
# Create DataFrame
df200 = pd.DataFrame(l200, columns=['Similar', 'Random', 'Distant'])
df200.insert(0, 'User id', users_kept)
df200.set_index("User id")

In [ ]:
l50 = []
np.random.seed(1)
for id in top_users[:50]:
    res = bev.knn_ROUGE(id, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, threshold = 0.13, k = 40, topx = 150)
    if res[0] is not None:
        l50.append(res[0]) 

In [ ]:
# Create DataFrame
df50 = pd.DataFrame(l50, columns=['Similar', 'Random', 'Distant'])
df50.insert(0, 'User id', top_users[:50])

In [ ]:
df200[['Random', 'Similar', 'Distant']].mean(), df50[['Random', 'Similar', 'Distant']].mean()

In [ ]:
df200['Type'] = '200 Random users'

In [ ]:
df50['Type'] = '50 Most active users'

In [ ]:
concat = pd.concat([df200, df50])
concat[['Similar', 'Random', 'Distant']] *= 100
concat

In [ ]:
df_melted_concat = concat.melt(id_vars=['User id', 'Type'], value_vars=['Similar', 'Random', 'Distant'],
                    var_name='Category', value_name='Value')

df_melted_concat['Category'] = pd.Categorical(
    df_melted_concat['Category'],
    categories=['Distant', 'Random', 'Similar'],
    ordered=True
)

mean_values = df_melted_concat.groupby(['Type', 'Category'])['Value'].mean().reset_index()
mean_values, df_melted_concat

In [ ]:
ax = sns.violinplot(data=df_melted_concat, x='Type', y='Value', hue='Category', inner='box', fill=False, cut=0, order=['50 Most active users', '200 Random users'])

offset = 0.267
xticks = ax.get_xticks()
category_offsets = {'Similar': offset, 'Random': 0, 'Distant': -offset}
type_to_x = {
    '50 Most active users': 0,  
    '200 Random users': 1       
}

for _, row in mean_values.iterrows():
    base_x = type_to_x[row['Type']]
    offset = category_offsets[row['Category']]
    x_pos = base_x + offset
    plt.scatter(x_pos, row['Value'], color='black', zorder=10, label='Mean' if _ == 0 else None, alpha=0.7)


# Avoid duplicate legend entries
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys())
plt.ylabel("ROUGE-2")
plt.xlabel("")

plt.tight_layout()
plt.savefig("vp_thd_top150.svg")
plt.show()


In [ ]:
bev.knn_ROUGE(208, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, threshold = 0, k = 40, topx = None)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# lemmatized comments
all_doc = filtered_rev_df['Comment']
vectorizer = TfidfVectorizer(ngram_range=(1, 1), min_df=5, max_df=0.8) # bigrams
vectors_unig = vectorizer.fit_transform(all_doc)
unig_ens = vectorizer.get_feature_names_out()

In [ ]:
test = []
test_r = []
test_b = []

np.random.seed(1)
for id in top_users[:50]:
    # res = bev.knn_ROUGE12(id, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, vectors_unig, unig_ens, threshold = 0.13, k = 40, topx = 150)
    res = bev.knn_ROUGE_prim(id, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, threshold = -1, k = 40, topx = None)  
    if res is not None:
        test.append(res[0])
        test_r.append(res[1])
        test_b.append(res[2]) 

In [ ]:
test2 = []
test_r2 = []
test_b2 = []
users_kept = []

np.random.seed(1)
random_users = np.random.choice(users_keeps, size=200, replace=False)

for id in random_users:
    # res = bev.knn_ROUGE12(id, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, vectors_unig, unig_ens, threshold = 0.13, k = 40, topx = 150)
    res = bev.knn_ROUGE_prim(id, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, threshold = -1, k = 40, topx = None)  
    if res[0] is not None:
        test2.append(res[0])
        test_r2.append(res[1]) 
        test_b2.append(res[2])
        users_kept.append(id)

In [ ]:
# Create DataFrame FOR ROUGE2
test_df = pd.DataFrame(test_r2, columns=['Similar users', 'Random users', 'Distant users'])
test_df.insert(0, 'User id', users_kept)
test_df['Type'] = '200 Random users'

test_dfr1 = pd.DataFrame(test_r, columns=['Similar users', 'Random users', 'Distant users'])
test_dfr1.insert(0, 'User id', top_users[:50])
test_dfr1['Type'] = '50 Most active users'

concat_r1 = pd.concat([test_df, test_dfr1])
concat_r1

df_melted_concat = concat_r1.melt(id_vars=['User id', 'Type'], value_vars=['Similar users', 'Random users', 'Distant users'],
                    var_name='Category', value_name='Value')

df_melted_concat['Category'] = pd.Categorical(
    df_melted_concat['Category'],
    categories=[ 'Similar users', 'Random users', 'Distant users'],
    ordered=True
)
df_melted_concat['Value'] = df_melted_concat['Value'] 

mean_values = df_melted_concat.groupby(['Type', 'Category'])['Value'].mean().reset_index()

In [ ]:
# rouge 2
df_melted_r2 = df_melted_concat
mean_val_r2 = mean_values

In [ ]:
# rouge1
df_melted_r1 = df_melted_concat
mean_val_r1 = mean_values

In [ ]:
# bleu
df_melted_b = df_melted_concat
mean_val_b = mean_values

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

fig, axs = plt.subplots(1, 3, figsize=(18, 6))  # 3 subplots
fig.suptitle("Distribution of per-user mean ROUGE, BLEU scores", fontsize=15, y=0.94)

offset = 0.267
category_offsets = {'Similar users': -offset, 'Random users': 0, 'Distant users': offset}
type_to_x = {'50 Most active users': 0, '200 Random users': 1}

def plot_violin(ax, data, mean_vals, title, ylabel):
    sns.violinplot(data=data, x='Type', y='Value', hue='Category',
                   inner='box', fill=False, cut=0,
                   order=['50 Most active users', '200 Random users'],
                   ax=ax)
    
    ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    for i, row in mean_vals.iterrows():
        base_x = type_to_x[row['Type']]
        offset = category_offsets[row['Category']]
        x_pos = base_x + offset
        ax.scatter(x_pos, row['Value'], color='black', zorder=10,
                   label='(mean)' if i == 0 else None, alpha=0.7)

    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), title="Prediction by", loc='upper left')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")

# ROUGE-1
plot_violin(axs[0], df_melted_r1, mean_val_r1, "", "ROUGE-1 (in %)")

# ROUGE-2
plot_violin(axs[1], df_melted_r2, mean_val_r2, "", "ROUGE-2 (in %)")

plot_violin(axs[2], df_melted_b, mean_val_b, "", "BLEU (in %)")
plt.tight_layout()
plt.subplots_adjust(top=0.88)
plt.savefig("VP_NOtfidf_NOtopx_RB.svg", format='svg')
plt.show()


In [ ]:
ax = sns.violinplot(data=df_melted_concat, x='Type', y='Value', hue='Category', inner='box', fill=False, cut=0, order=['50 Most active users', '200 Random users'])

offset = 0.267
xticks = ax.get_xticks()

category_offsets = {'Similar': offset, 'Random': 0, 'Distant': -offset}
type_to_x = {
    '50 Most active users': 0,  
    '200 Random users': 1       
}

for _, row in mean_values.iterrows():
    base_x = type_to_x[row['Type']]
    offset = category_offsets[row['Category']]
    x_pos = base_x + offset
    plt.scatter(x_pos, row['Value'], color='black', zorder=10, label='Mean' if _ == 0 else None, alpha=0.7)


# Avoid duplicate legend entries
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys())
plt.ylabel("ROUGE-2")
plt.xlabel("")

plt.tight_layout()
# plt.savefig("vp_thd_NOtop_ROUGE2.svg")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

fig, axs = plt.subplots(1, 2, figsize=(13, 6))  # 3 subplots
fig.suptitle("Distribution of per-user mean ROUGE scores", fontsize=15, y=0.94)

offset = 0.267
category_offsets = {'Similar users': -offset, 'Random users': 0, 'Distant users': offset}
type_to_x = {'50 Most active users': 0, '200 Random users': 1}

def plot_violin(ax, data, mean_vals, title, ylabel):
    sns.violinplot(data=data, x='Type', y='Value', hue='Category',
                   inner='box', fill=False, cut=0,
                   order=['50 Most active users', '200 Random users'],
                   ax=ax)
    
    ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    for i, row in mean_vals.iterrows():
        base_x = type_to_x[row['Type']]
        offset = category_offsets[row['Category']]
        x_pos = base_x + offset
        ax.scatter(x_pos, row['Value'], color='black', zorder=10,
                   label='(mean)' if i == 0 else None, alpha=0.7)

    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), title="Prediction by", loc='upper left')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")

# ROUGE-1
plot_violin(axs[0], df_melted_r1, mean_val_r1, "", "ROUGE-1 (in %)")

# ROUGE-2
plot_violin(axs[1], df_melted_r2, mean_val_r2, "", "ROUGE-2 (in %)")

plt.tight_layout()
plt.subplots_adjust(top=0.88)
plt.savefig("VP_tfidf_topx_rouges.svg", format='svg')
plt.show()


In [ ]:
# Create DataFrame
test_df_r2 = pd.DataFrame(test_r2, columns=['Similar', 'Random', 'Distant'])
test_df_r2.insert(0, 'User id', users_kept)
test_df_r2.set_index("User id")

In [ ]:
test_df_r2['Type'] = '200 Random users'

In [ ]:
# Create DataFrame
test_df = pd.DataFrame(test, columns=['Similar', 'Random', 'Distant'])
test_df.insert(0, 'User id', top_users[:50])

In [ ]:
test_df_r = pd.DataFrame(test_r, columns=['Similar', 'Random', 'Distant'])
test_df_r.insert(0, 'User id', top_users[:50])

In [ ]:
test_df_r['Type'] = '50 Most active users'

In [ ]:
concat_r = pd.concat([test_df_r, test_df_r2])
concat_r

In [ ]:
df_melted_concat = concat_r.melt(id_vars=['User id', 'Type'], value_vars=['Similar', 'Random', 'Distant'],
                    var_name='Category', value_name='Value')

df_melted_concat['Category'] = pd.Categorical(
    df_melted_concat['Category'],
    categories=['Distant', 'Random', 'Similar'],
    ordered=True
)

mean_values = df_melted_concat.groupby(['Type', 'Category'])['Value'].mean().reset_index()
mean_values, df_melted_concat

In [ ]:
ax = sns.violinplot(data=df_melted_concat, x='Type', y='Value', hue='Category', inner='box', fill=False, cut=0, order=['50 Most active users', '200 Random users'])

offset = 0.267
xticks = ax.get_xticks()
category_offsets = {'Similar': offset, 'Random': 0, 'Distant': -offset}
type_to_x = {
    '50 Most active users': 0,  
    '200 Random users': 1       
}

for _, row in mean_values.iterrows():
    base_x = type_to_x[row['Type']]
    offset = category_offsets[row['Category']]
    x_pos = base_x + offset
    plt.scatter(x_pos, row['Value'], color='black', zorder=10, label='Mean' if _ == 0 else None, alpha=0.7)


# Avoid duplicate legend entries
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys())
plt.ylabel("ROUGE-1")
plt.xlabel("")

plt.tight_layout()
plt.savefig("vp_thd_top_ROUGE1.svg")
plt.show()


In [ ]:
concat_r.groupby("Type").mean()

Test TOPX ROUGE 2

In [ ]:
# no tf idf

lst_topx = []
lst_topx_r = []

np.random.seed(1)
for k in range(1, 300):
    print(k)
    lst = []
    lst_r = []
    for id in top_users[:50]:
        res = bev.knn_ROUGE(id, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, threshold = -1, k = 40, topx = k)
        if res is not None:
            lst.append(res[0])
            lst_r.append(res[1]) 
    lst_topx.append(np.mean(lst))
    lst_topx_r.append(np.mean(lst_r))

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import numpy as np

lst_topx = []
lst_topx_r = []
kkeep = []

np.random.seed(1)
# random_users = np.random.choice(users_keeps, size=200, replace=False)
def process_user(id_k):
    id, k = id_k
    res = bev.knn_ROUGE(
        id, matrix_ratings, mask_ratings, cos_sim_matrix,
        users_table, games_table, comments_all, vectors, bigrams_ens,
        threshold=-1, k=40, topx=k
    )
    return res if res is not None else (None, None)

num_workers = 9  # Or use os.cpu_count()

for k in range(1, 301, 5):
    print(k)
    lst = []
    lst_r = []

    args = [(id, k) for id in top_users[:50]]

    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        results = executor.map(process_user, args)

    for res0, res1 in results:
        if res0 is not None:
            lst.append(res0)
            lst_r.append(res1)

    lst_topx.append(np.mean(lst, axis=0))
    lst_topx_r.append(np.mean(lst_r, axis=0))


In [ ]:
np.array(lst_topx)[:,0]

In [ ]:
# rouge 2 only
lst_topx_200 = lst_topx

In [ ]:
# rouge 2 only
lst_topx_50 = lst_topx

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.ndimage import uniform_filter1d  

palette = sns.color_palette("deep")
colors = {"Similar": palette[0], "Random": palette[1], "Distant": palette[2]}
x = range(1, 300, 5)

data_sets = [
    (lst_topx_200, "200 Random users ROUGE score"),
    (lst_topx_50, "50 Most active users ROUGE score"),
]

fig, axs = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("ROUGE scores evolution with top X bigrams, using TF-IDF", fontsize=14, y=0.96)

for ax, (data, title) in zip(axs, data_sets):
    data = np.array(data) * 100
    for idx, label in enumerate(["Similar", "Random", "Distant"]):
        smoothed = uniform_filter1d(data[:, idx], size=3) 
        ax.plot(x, smoothed, c=colors[label], label=label)
    ax.axvline(x=150, color='gray', linestyle='--')
    ax.set_xlabel("x")
    ax.set_ylabel("ROUGE-2 (in %)")
    ax.set_title(title)
    ax.legend(title="Prediction by")

plt.savefig("Courbe_topx.svg", format='svg')
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# lemmatized comments
all_doc = filtered_rev_df['Comment']
vectorizer = TfidfVectorizer(ngram_range=(1, 1), min_df=5, max_df=0.8) # bigrams
vectors_unig = vectorizer.fit_transform(all_doc)
unig_ens = vectorizer.get_feature_names_out()

In [ ]:
np.random.seed(1)
bev.knn_ROUGE12(208, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, vectors_unig,unig_ens , threshold = 0, k = 40, topx = None)

In [ ]:
np.random.seed(1)
bev.knn_ROUGE12(208, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, vectors_unig,unig_ens, threshold = 0.13, k = 40, topx = None)

In [ ]:
from sacrebleu import BLEU

bleu = BLEU(max_ngram_order=1,effective_order=True)
bleu.sentence_score(hypothesis="je suis la je suis la", references=["je suis la"]).score

In [ ]:
bev.knn_ROUGE_prim(208, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, threshold = 0, k = 40, topx = None)

In [ ]:
test = []
test_r = []
test_b = []

np.random.seed(1)
for id in top_users[:50]:
    res = bev.knn_ROUGE_prim(id, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, threshold = 0, k = 40, topx = None)
    if res is not None:
        test.append(res[0])
        test_r.append(res[1]) 
        test_b.append(res[2])

In [ ]:
test2 = []
test_r2 = []
test_b2 = []
users_kept = []

np.random.seed(1)
random_users = np.random.choice(users_keeps, size=200, replace=False)

for id in random_users:
    res = bev.knn_ROUGE_prim(id, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, threshold = 0, k = 40, topx = None)
    if res[0] is not None:
        test2.append(res[0])
        test_r2.append(res[1]) 
        test_b2.append(res[2])
        users_kept.append(id)

In [ ]:
# Create DataFrame FOR ROUGE2
test_df = pd.DataFrame(test2, columns=['Similar', 'Random', 'Distant'])
test_df.insert(0, 'User id', users_kept)
test_df['Type'] = '200 Random users'

test_dfr1 = pd.DataFrame(test, columns=['Similar', 'Random', 'Distant'])
test_dfr1.insert(0, 'User id', top_users[:50])
test_dfr1['Type'] = '50 Most active users'

concat_r1 = pd.concat([test_df, test_dfr1])
concat_r1

df_melted_concat = concat_r1.melt(id_vars=['User id', 'Type'], value_vars=['Similar', 'Random', 'Distant'],
                    var_name='Category', value_name='Value')

df_melted_concat['Category'] = pd.Categorical(
    df_melted_concat['Category'],
    categories=['Distant', 'Random', 'Similar'],
    ordered=True
)

mean_values = df_melted_concat.groupby(['Type', 'Category'])['Value'].mean().reset_index()

In [ ]:
ax = sns.violinplot(data=df_melted_concat, x='Type', y='Value', hue='Category', inner='box', fill=False, cut=0, order=['50 Most active users', '200 Random users'])

offset = 0.267
xticks = ax.get_xticks()
category_offsets = {'Similar': offset, 'Random': 0, 'Distant': -offset}
type_to_x = {
    '50 Most active users': 0,  
    '200 Random users': 1       
}

for _, row in mean_values.iterrows():
    base_x = type_to_x[row['Type']]
    offset = category_offsets[row['Category']]
    x_pos = base_x + offset
    plt.scatter(x_pos, row['Value'], color='black', zorder=10, label='Mean' if _ == 0 else None, alpha=0.7)


# Avoid duplicate legend entries
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys())
plt.ylabel("ROUGE-2")
plt.xlabel("")

plt.tight_layout()
plt.savefig("vp_NOthd_NOtop_ROUGE2_last.svg")
plt.show()


In [ ]:
np.random.seed(1)
bev.knn_ROUGE12(208, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, vectors_unig,unig_ens , threshold = 0.13, k = 40, topx = 150)

In [ ]:
# generate comment user random, best score

users_kept = []
r1 = []
r2 = []

np.random.seed(1)
random_users = np.random.choice(users_keeps, size=200, replace=False)

for id in random_users:
    print("users ",id)
    res = bev.knn_ROUGE12(208, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, vectors_unig,unig_ens , threshold = 0.13, k = 40, topx = 150)
    if res[0] is not None:
        r1.append(res[0])
        r2.append(res[1])

        users_kept.append(id)

In [ ]:
import textwrap

long_text = "This is a very long sentence that should be wrapped neatly when printed in a Jupyter Notebook for better readability."

print(textwrap.fill(long_text, width=80))

In [ ]:
bev.knn_ROUGE_prim(208, matrix_ratings, mask_ratings, cos_sim_matrix, users_table, games_table, comments_all, vectors, bigrams_ens, threshold = 0, k = 40, topx = None)